# Apigee LLM Budget Control (Buffered + SSE Streaming): Interactive Demo

This notebook guides you through an interactive live demonstration of Apigee's micro-dollar (`1/1,000,000` USD) budget controls across **both buffered (`application/json`) and streaming Server-Sent Events (`text/event-stream`)** LLM APIs using `LLMTokenQuota` and `<EventFlow>`.

We will showcase:
1. **Unified Buffered + SSE Micro-Dollar Wallet**: How buffered (`:generateContent`) and streaming SSE (`:streamGenerateContent?alt=sse`) calls across multiple models (`gemini-2.5-flash`, `gemini-2.5-pro`) deduct from **one isolated monthly wallet per Developer App**.
2. **Real-Time SSE Streaming with Single Terminal Charge**: How Apigee streams SSE chunks immediately with zero buffering while executing `LTQ-CountBudget` and `DC-CollectCostMetrics` **exactly once** on the terminal usage chunk.
3. **Consistent Pre-Flight Telemetry Contract**: How all 7 `x-llm-*` headers are emitted identically across Buffered (`200`), Streaming SSE (`200`), and Over-Budget (`429`) responses.
4. **Cross-Mode Quota Blocking & Tenant Isolation**: How exhausting a team's budget blocks **both** future SSE and Buffered calls pre-flight (`HTTP 429`) with zero backend LLM spend, without affecting other teams.

---

## Scenario Design

We provision two Developer Apps backed by tiered Apigee API Products:
* **Team A (Marketing App)**: Low monthly budget of **`$0.02`** (`20,000` micro-dollars) to quickly trigger cross-mode `429` quota enforcement.
* **Team B (Engineering App)**: Operational monthly budget of **`$1.00`** (`1,000,000` micro-dollars) for continuous workloads.

## Setup & Reset (Run this cell to start or reset the demo)

Running the cell below will:
1. Recreate fresh **Marketing** and **Engineering** Developer Apps in Apigee with **`0` micro-dollar** quota balances via `setup-demo-resources.sh`.
2. Load the new API keys and environment settings into this notebook's memory.
3. Define `send_buffered(...)`, `send_sse(...)`, and `print_session_ledger()` helpers for real-time HTTP/SSE inspection.

**You can re-run this cell at any time during a presentation to reset all quota counters to zero.**

In [ ]:
import datetime
import http.client
import json
import os
from pathlib import Path
import ssl
import subprocess
import time

# Automatically locate the llm-budget-control-sse directory whether Jupyter started in repo root or sample dir
SAMPLE_DIR = next(
    (p for p in [Path.cwd(), Path.cwd() / "apigee-llm-budget-control-sse", Path.cwd() / "llm-budget-control-sse"] if (p / "apiproxy" / "llm-budget-control-sse.xml").exists()),
    Path.cwd(),
)
assert (SAMPLE_DIR / "setup-demo-resources.sh").exists(), f"Could not find setup-demo-resources.sh in {SAMPLE_DIR}"

print(f"Resetting demo Developer Apps & quota wallets in {SAMPLE_DIR.name} ...")
result = subprocess.run(
    ["bash", "setup-demo-resources.sh"],
    capture_output=True,
    text=True,
    cwd=str(SAMPLE_DIR),
)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("setup-demo-resources.sh failed")

# Parse configuration from env.sh and fresh keys from demo-keys.env
config = {}
for filename in ("env.sh", "demo-keys.env"):
    with open(SAMPLE_DIR / filename, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if line.startswith("export "):
                line = line[len("export "):].strip()
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                config[k.strip()] = v.strip().strip('"').strip("'")

project_id = config["PROJECT"]
apigee_host = config["APIGEE_HOST"]
region = config.get("REGION") or "us-west1"
if region == "REGION_TO_SET":
    region = "us-west1"
marketing_key = config["MARKETING_KEY"]
engineering_key = config["ENGINEERING_KEY"]
base_path = "/v2/samples/llm-budget-control-sse"

TELEMETRY_HEADERS = [
    "x-llm-provider",
    "x-llm-model",
    "x-llm-streaming",
    "x-llm-budget-limit-micro",
    "x-llm-budget-used-before-request-micro",
    "x-llm-budget-remaining-before-request-micro",
    "x-llm-budget-reset-utc",
]

session_ledger = []


def _fmt_usd(micro_str: str) -> str:
    try:
        val = int(micro_str)
        return f"{val:,} µ$ (${val / 1_000_000:.6f})"
    except (TypeError, ValueError):
        return str(micro_str)


def _print_telemetry(status: int, reason: str, headers_map: dict):
    print(f"=== HTTP STATUS: {status} {reason} ===")
    print("=== APIGEE TELEMETRY HEADERS ===")
    for h in TELEMETRY_HEADERS:
        val = headers_map.get(h, "<missing>")
        if h.endswith("-micro") and val != "<missing>":
            print(f"  {h:44s}: {_fmt_usd(val)}")
        elif h == "x-llm-budget-reset-utc" and val.isdigit():
            dt = datetime.datetime.fromtimestamp(int(val) / 1000, tz=datetime.timezone.utc)
            print(f"  {h:44s}: {val} ({dt.strftime('%Y-%m-%d %H:%M:%S UTC')})")
        else:
            print(f"  {h:44s}: {val}")


def _record_ledger(team: str, mode: str, model: str, status: int, headers_map: dict):
    session_ledger.append({
        "step": len(session_ledger) + 1,
        "team": team,
        "mode": mode,
        "model": headers_map.get("x-llm-model", model),
        "status": status,
        "streaming_hdr": headers_map.get("x-llm-streaming", ""),
        "limit_micro": headers_map.get("x-llm-budget-limit-micro", ""),
        "used_before_micro": headers_map.get("x-llm-budget-used-before-request-micro", ""),
        "remaining_before_micro": headers_map.get("x-llm-budget-remaining-before-request-micro", ""),
    })


def send_buffered(
    team: str,
    api_key: str,
    model: str,
    prompt: str,
    thinking_budget: int | None = None,
    include_provider_header: bool = True,
    settle_seconds: float = 2.5,
):
    """Send a standard buffered (:generateContent) request through Apigee."""
    path = f"{base_path}/v1/projects/{project_id}/locations/{region}/publishers/google/models/{model}:generateContent"
    payload = {"contents": [{"role": "user", "parts": [{"text": prompt}]}]}
    if thinking_budget is not None:
        payload["generationConfig"] = {"thinkingConfig": {"thinkingBudget": thinking_budget}}

    req_headers = {
        "x-apikey": api_key,
        "Content-Type": "application/json",
        "Connection": "close",
    }
    if include_provider_header:
        req_headers["x-llm-provider"] = "google"

    conn = http.client.HTTPSConnection(apigee_host, timeout=90, context=ssl.create_default_context())
    conn.request("POST", path, body=json.dumps(payload).encode("utf-8"), headers=req_headers)
    resp = conn.getresponse()
    headers_map = {k.lower(): v for k, v in resp.getheaders()}
    raw_body = resp.read().decode("utf-8", errors="replace")
    conn.close()

    _print_telemetry(resp.status, resp.reason, headers_map)
    _record_ledger(team, "Buffered", model, resp.status, headers_map)

    print("\n=== BUFFERED RESPONSE ===")
    try:
        data = json.loads(raw_body)
    except json.JSONDecodeError:
        print(raw_body)
        return

    if resp.status == 200:
        parts = data.get("candidates", [{}])[0].get("content", {}).get("parts", [])
        text = "".join(p.get("text", "") for p in parts if not p.get("thought"))
        usage = data.get("usageMetadata", {})
        print(text.strip())
        print(
            f"\n[Token Usage -> prompt: {usage.get('promptTokenCount', 0)}, "
            f"candidates: {usage.get('candidatesTokenCount', 0)}, "
            f"thoughts: {usage.get('thoughtsTokenCount', 0)}]"
        )
        if settle_seconds > 0:
            time.sleep(settle_seconds)
    else:
        print(json.dumps(data, indent=2))


def send_sse(
    team: str,
    api_key: str,
    model: str,
    prompt: str,
    thinking_budget: int | None = None,
    include_provider_header: bool = False,
    settle_seconds: float = 2.5,
):
    """Send a streaming SSE (:streamGenerateContent?alt=sse) request and render chunks live."""
    path = (
        f"{base_path}/v1/projects/{project_id}/locations/{region}"
        f"/publishers/google/models/{model}:streamGenerateContent?alt=sse"
    )
    payload = {"contents": [{"role": "user", "parts": [{"text": prompt}]}]}
    if thinking_budget is not None:
        payload["generationConfig"] = {"thinkingConfig": {"thinkingBudget": thinking_budget}}

    req_headers = {
        "x-apikey": api_key,
        "Accept": "text/event-stream",
        "Content-Type": "application/json",
        "Connection": "close",
    }
    if include_provider_header:
        req_headers["x-llm-provider"] = "google"

    conn = http.client.HTTPSConnection(apigee_host, timeout=90, context=ssl.create_default_context())
    conn.request("POST", path, body=json.dumps(payload).encode("utf-8"), headers=req_headers)
    resp = conn.getresponse()
    headers_map = {k.lower(): v for k, v in resp.getheaders()}

    _print_telemetry(resp.status, resp.reason, headers_map)
    _record_ledger(team, "SSE Stream", model, resp.status, headers_map)

    if resp.status != 200:
        err_body = resp.read().decode("utf-8", errors="replace")
        conn.close()
        print("\n=== GATEWAY PRE-FLIGHT BLOCK (NO STREAM OPENED) ===")
        try:
            print(json.dumps(json.loads(err_body), indent=2))
        except json.JSONDecodeError:
            print(err_body)
        return

    print(f"\n=== LIVE SSE STREAM (Content-Type: {headers_map.get('content-type', '')}) ===")
    chunk_count = 0
    terminal_finish_reason = None
    terminal_usage = {}

    while True:
        raw_line = resp.readline()
        if not raw_line:
            break
        line = raw_line.decode("utf-8", errors="replace").strip()
        if not line.startswith("data:"):
            continue
        payload_str = line[5:].strip()
        if not payload_str or payload_str == "[DONE]":
            continue
        chunk_count += 1
        try:
            chunk_json = json.loads(payload_str)
        except json.JSONDecodeError:
            continue

        cand = (chunk_json.get("candidates") or [{}])[0]
        for part in cand.get("content", {}).get("parts", []):
            if not part.get("thought") and part.get("text"):
                print(part["text"], end="", flush=True)

        if cand.get("finishReason"):
            terminal_finish_reason = cand["finishReason"]
            terminal_usage = chunk_json.get("usageMetadata", {})

    conn.close()
    print(
        f"\n\n[SSE EventFlow Summary -> chunks: {chunk_count}, "
        f"terminal finishReason: {terminal_finish_reason}, "
        f"prompt: {terminal_usage.get('promptTokenCount', 0)}, "
        f"candidates: {terminal_usage.get('candidatesTokenCount', 0)}, "
        f"thoughts: {terminal_usage.get('thoughtsTokenCount', 0)}]"
    )
    if settle_seconds > 0:
        time.sleep(settle_seconds)


def print_session_ledger():
    """Render an aligned summary table of every Buffered & SSE request in this session."""
    header = (
        f"{'#':>2} | {'Team':<11} | {'Mode':<10} | {'Model':<16} | {'HTTP':>4} | "
        f"{'Streaming':<9} | {'Limit (µ$)':>12} | {'Used Before (µ$)':>16} | {'Remaining Before (µ$)':>21}"
    )
    print(header)
    print("-" * len(header))
    for row in session_ledger:
        print(
            f"{row['step']:>2} | {row['team']:<11} | {row['mode']:<10} | {row['model']:<16} | "
            f"{row['status']:>4} | {row['streaming_hdr']:<9} | {row['limit_micro']:>12} | "
            f"{row['used_before_micro']:>16} | {row['remaining_before_micro']:>21}"
        )


print("\n--- Loaded Configuration ---")
print(f"Project ID                  : {project_id}")
print(f"Apigee Host                 : {apigee_host}")
print(f"Vertex AI Region            : {region}")
print(f"Marketing Key ($0.02 cap)   : {marketing_key[:8]}...")
print(f"Engineering Key ($1.00 cap) : {engineering_key[:8]}...")
print("\nWaiting 15 seconds for fresh API Keys to propagate to Apigee runtime pods...")
time.sleep(15)
print("Quota Reset Complete! Ready for the interactive demo.")

## Step 1: Engineering Team — Buffered Request (`gemini-2.5-flash`)

We start by sending a standard **buffered** (`:generateContent`) request using the **Engineering App** (`$1.00` = `1,000,000 µ$` monthly budget):
* `x-llm-streaming` resolves to `"false"`.
* `x-llm-budget-used-before-request-micro` starts at **`0 µ$`** (`$0.000000`).

In [ ]:
send_buffered(
    team="Engineering",
    api_key=engineering_key,
    model="gemini-2.5-flash",
    prompt="Explain an API gateway in one concise sentence.",
)

## Step 2: Engineering Team — Live SSE Streaming (`gemini-2.5-pro`, Auto-Detected Provider)

Next, we send a **streaming Server-Sent Events (SSE)** request (`:streamGenerateContent?alt=sse`) using the **same Engineering App**, switching to the higher-tier **`gemini-2.5-pro`** model **without** passing an `x-llm-provider` header:
* Apigee's OOTB `EV-ExtractRequest` + `AM-InitRequest` automatically detects `x-llm-provider: google` and `x-llm-streaming: true` directly from the URI.
* Notice `x-llm-budget-used-before-request-micro` already reflects Step 1's buffered `gemini-2.5-flash` spend!
* Chunks stream live to the client while `<EventFlow>` triggers `LTQ-CountBudget` **once** on the terminal `finishReason: STOP` chunk.

In [ ]:
send_sse(
    team="Engineering",
    api_key=engineering_key,
    model="gemini-2.5-pro",
    prompt="List three benefits of Server-Sent Events for streaming LLM responses in two short sentences.",
    thinking_budget=128,
    include_provider_header=False,  # Auto-detected from Vertex AI URI!
)

## Step 3: Engineering Team — Verify Cross-Mode & Cross-Model Wallet Accumulation

Let's send one more quick request on **`gemini-2.5-flash`** and inspect `x-llm-budget-used-before-request-micro`:
* Because `<Identifier ref="developer.app.id"/>` and `<LLMModelSource>all-models</LLMModelSource>` scope the quota counter to the Developer App across **all** models and delivery modes, the opening balance now includes **both** Step 1 (Buffered Flash) and Step 2 (Streaming SSE Pro)!

In [ ]:
send_buffered(
    team="Engineering",
    api_key=engineering_key,
    model="gemini-2.5-flash",
    prompt="Say 'Unified wallet verified!' and give a 6-word definition of Server-Sent Events.",
)

## Step 4: Marketing Team (`$0.02` Budget) — First Request Succeeds

Now we switch to the **Marketing App**, whose API Product (`llm-budget-sse-marketing-product`) defines a tight monthly budget of **`20,000 µ$`** (**`$0.02`**):
* First, Marketing sends a short buffered request on `gemini-2.5-flash`.
* This costs under `1,000 µ$`, leaving > `19,000 µ$` available in Marketing's wallet.

In [ ]:
send_buffered(
    team="Marketing",
    api_key=marketing_key,
    model="gemini-2.5-flash",
    prompt="Write a catchy 8-word slogan for a solar smartwatch.",
    thinking_budget=0,
)

## Step 5: Marketing Team — SSE Stream Succeeds & Exhausts Remaining Budget

Next, Marketing streams a comprehensive 550-word product newsletter on **`gemini-2.5-pro`** via SSE (`:streamGenerateContent?alt=sse`):
* **Pre-Flight Check (`LTQ-EnforceBudget`)**: Marketing still has > `19,900 µ$` remaining (`> 0`), so Apigee allows the SSE stream to open (`HTTP 200`, `x-llm-streaming: true`).
* **Terminal `<EventFlow>` Charge (`LTQ-CountBudget`)**: Once the stream finishes, `JS-CalculateCost` computes the full prompt + output + thinking token cost (exceeding the remaining `20,000 µ$` cap) and updates the quota counter with `continueOnError="true"` so the already-paid stream completes cleanly—leaving Marketing's wallet **exhausted**.

In [ ]:
send_sse(
    team="Marketing",
    api_key=marketing_key,
    model="gemini-2.5-pro",
    prompt="Write a comprehensive 550-word product launch newsletter highlighting why outdoor runners and mountaineers love our new solar-powered smartwatch, including technical specs, battery endurance, GPS accuracy, and customer testimonials.",
    thinking_budget=2048,
)

## Step 6a: Marketing Team — Subsequent SSE Stream Blocked Pre-Flight (`429`)

Marketing now tries to open another **SSE stream**, even downgrading to the cheaper **`gemini-2.5-flash`** model:
* `LTQ-EnforceBudget` immediately rejects the request in Request `PreFlow` with **`HTTP 429 Too Many Requests`** (`RESOURCE_EXHAUSTED`).
* Notice all 7 `x-llm-*` telemetry headers are still returned (`x-llm-streaming: true`, `x-llm-budget-remaining-before-request-micro: 0`), **no SSE stream is opened**, and **zero backend Vertex AI cost** is incurred.

In [ ]:
send_sse(
    team="Marketing",
    api_key=marketing_key,
    model="gemini-2.5-flash",
    prompt="Stream a one-paragraph summary of solar battery technology.",
)

## Step 6b: Marketing Team — Switching to Buffered Mode Is Also Blocked (`429`)

What if Marketing tries bypassing the SSE block by switching back to standard buffered `:generateContent`?
* Because Buffered and SSE share the exact same `llm_budget_quota` counter, the buffered call is **also blocked pre-flight** (`HTTP 429`, `x-llm-streaming: false`, `x-llm-budget-remaining-before-request-micro: 0`).

In [ ]:
send_buffered(
    team="Marketing",
    api_key=marketing_key,
    model="gemini-2.5-flash",
    prompt="Explain solar battery technology in one sentence.",
)

## Step 7: Tenant Isolation — Engineering Team SSE Still Streams Normally (`200 OK`)

To prove strict per-tenant isolation (`<Identifier ref="developer.app.id"/>`), we immediately send a live **SSE streaming** request using the **Engineering App**:
* Even though Marketing is blocked across both modes, Engineering streams normally with plenty of its `$1.00` budget remaining.

In [ ]:
send_sse(
    team="Engineering",
    api_key=engineering_key,
    model="gemini-2.5-flash",
    prompt="Explain in one sentence why tenant-isolated AI budgets are essential for enterprise platforms.",
)

## Step 8 (Bonus): Drop-In `google-genai` Python SDK Streaming

Application teams do not need to write custom SSE parsers—they can point Google's official `google-genai` SDK directly at the Apigee proxy `base_url` with `x-apikey`!
* Notice the client passes `Credentials(token="apigee-managed")`: client apps need **zero Google Cloud IAM permissions** because Apigee authenticates the caller via `x-apikey` and injects Vertex AI IAM credentials southbound.

In [ ]:
import httpx
from google import genai
from google.genai import types
from google.oauth2.credentials import Credentials

sdk_headers = {}


def _capture_apigee_headers(response: httpx.Response):
    sdk_headers.clear()
    for k, v in response.headers.items():
        if k.lower().startswith("x-llm-"):
            sdk_headers[k.lower()] = v


genai_client = genai.Client(
    vertexai=True,
    project=project_id,
    location=region,
    credentials=Credentials(token="apigee-managed"),
    http_options=types.HttpOptions(
        base_url=f"https://{apigee_host}{base_path}",
        headers={"x-apikey": engineering_key},
        httpx_client=httpx.Client(event_hooks={"response": [_capture_apigee_headers]}, timeout=60.0),
    ),
)

print("=== google-genai SDK: generate_content_stream() ===")
for chunk in genai_client.models.generate_content_stream(
    model="gemini-2.5-flash",
    contents="Give two short bullet points on why Apigee + google-genai SDK makes AI governance effortless.",
):
    if chunk.text:
        print(chunk.text, end="", flush=True)

print("\n\n=== Captured Apigee x-llm-* Headers from SDK Stream ===")
for h in TELEMETRY_HEADERS:
    print(f"  {h:44s}: {sdk_headers.get(h, '<missing>')}")

## Step 9: Full Session Budget Ledger

Run the cell below to view the complete progression of opening balances, modes (`Buffered` vs `SSE Stream`), models, and HTTP statuses recorded across this demo session:

In [ ]:
print_session_ledger()

## Wrap-Up & Key Architectural Takeaways

### 1. Unified Two-Phase Quota Across Buffered & `<EventFlow>` SSE
* **Request `PreFlow` (`LTQ-EnforceBudget`, `EnforceOnly`, `100% OOTB`)**: Authenticates `x-apikey`, extracts model/provider/SSE flags, loads KVM pricing, and blocks exhausted apps pre-flight (`429`) before any backend connection or SSE stream is opened.
* **Response / Stream Accounting (`LTQ-CountBudget`, `CountOnly`)**:
  * **Buffered (`application/json`)**: Charges once in Proxy `PostFlow`.
  * **Streaming (`text/event-stream`)**: Streams every `data:` event immediately through Target `<EventFlow>`, while `JS-CalculateCost` sets `should_record_cost = "true"` **only on the terminal usage chunk** so `LTQ-CountBudget` and `DC-CollectCostMetrics` execute **exactly once** per stream.

### 2. Why Telemetry Headers Report Opening Balance
In HTTP/SSE, response headers are flushed to the client **before** the first token streams—when final completion token counts do not yet exist. Exposing `x-llm-budget-used-before-request-micro` and `x-llm-budget-remaining-before-request-micro` provides a 100% deterministic, apples-to-apples header contract across Buffered (`200`), SSE (`200`), and Blocked (`429`) calls, while exact per-request token counts and micro-dollar costs are recorded in **Apigee Analytics** (`dc_llm_*`).

### 3. Metadata-Driven API Product Budgets & Proxy Chaining
* Change `budget_limit_micro_dollars`, `budget_interval`, or `budget_timeunit` directly on the **Apigee API Product** in the UI or API—zero proxy code changes or redeployments required.
* Existing Apigee proxies (e.g., Model Armor security or semantic caching) can chain to `/v2/samples/llm-budget-control-sse` in-memory via `<LocalTargetConnection>` so all proxies share one unified per-App budget counter.